# Copy Sample Data

This notebook copies sample and reference data into the bronze lakehouse Ingest/ folders to prepare for pipeline execution.

**What it does:**
1. **(Optional)** Copies SampleData/ and ReferenceData/ from a production workspace's bronze lakehouse to the current workspace
2. Copies all Clinical FHIR files from SampleData/ → Ingest/
3. Copies all DICOM imaging files
4. Copies Claims CCLF files
5. Copies SDOH CSV/XLSX files
6. Loads ZipToFipsMapping into Silver Delta table

**Configuration:**
- PROD_WORKSPACE_ID: Set to copy data from another workspace (leave empty if data is already local)

**Idempotent:** Safe to re-run — checks for existing files before copying.

In [ ]:
# =============================================================================
# WORKSPACE & LAKEHOUSE DISCOVERY
# =============================================================================

from notebookutils import mssparkutils
from sempy import fabric

# Get workspace context dynamically
ctx = mssparkutils.runtime.context
WORKSPACE_ID = ctx['currentWorkspaceId']
WORKSPACE_NAME = ctx['currentWorkspaceName'].strip()

# Discover lakehouses by keyword in name
df_items = fabric.list_items(workspace=WORKSPACE_ID)
lakehouses = df_items[df_items['Type'] == 'Lakehouse']

bronze_match = lakehouses[lakehouses['Display Name'].str.contains('bronze', case=False)]
silver_match = lakehouses[lakehouses['Display Name'].str.contains('silver', case=False)]

if len(bronze_match) != 1:
    raise RuntimeError(f"Expected 1 bronze lakehouse, found {len(bronze_match)}: {bronze_match['Display Name'].tolist()}")
if len(silver_match) != 1:
    raise RuntimeError(f"Expected 1 silver lakehouse, found {len(silver_match)}: {silver_match['Display Name'].tolist()}")

BRONZE_LAKEHOUSE_NAME = bronze_match.iloc[0]['Display Name']
SILVER_LAKEHOUSE_NAME = silver_match.iloc[0]['Display Name']
BRONZE_LAKEHOUSE_ID = bronze_match.iloc[0]['Id']
SILVER_LAKEHOUSE_ID = silver_match.iloc[0]['Id']

# Build root paths using GUID-based format (avoids name-resolution issues)
# Dynamically detect OneLake endpoint from lakehouse properties
bronze_props = mssparkutils.lakehouse.get(BRONZE_LAKEHOUSE_NAME, WORKSPACE_ID)
bronze_abfs = bronze_props['properties']['abfsPath']
ENDPOINT_URI = bronze_abfs.split("@")[1].split("/")[0]

bronze_root_path = f"abfss://{WORKSPACE_ID}@{ENDPOINT_URI}/{BRONZE_LAKEHOUSE_ID}"
silver_root_path = f"abfss://{WORKSPACE_ID}@{ENDPOINT_URI}/{SILVER_LAKEHOUSE_ID}"

print(f"\u2713 Workspace: {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"\u2713 Endpoint: {ENDPOINT_URI}")
print(f"\u2713 Bronze: {BRONZE_LAKEHOUSE_NAME} ({BRONZE_LAKEHOUSE_ID})")
print(f"\u2713 Silver: {SILVER_LAKEHOUSE_NAME} ({SILVER_LAKEHOUSE_ID})")
print(f"\u2713 Bronze path: {bronze_root_path}")

# Shared helper for idempotency checks
def file_exists(path):
    try:
        mssparkutils.fs.ls(path)
        return True
    except Exception:
        return False

def file_was_copied(relative_path):
    """Check if a file exists in Ingest, Process, or Failed folders."""
    for stage in ["Ingest", "Process", "Failed"]:
        full_path = f"{bronze_root_path}/Files/{stage}/{relative_path}"
        if file_exists(full_path):
            return True
    return False

In [ ]:
# =============================================================================
# COPY SAMPLEDATA & REFERENCEDATA FROM PRODUCTION WORKSPACE
# =============================================================================
# Set PROD_WORKSPACE_ID to copy data from prod bronze to current bronze.
# Leave empty to skip this step.

from datetime import datetime
PROD_WORKSPACE_ID = "5bf643a0-8ee0-4c6b-8476-c8fa9a243d10"  # e.g. "3fe06b16-a473-4e6e-a3fe-6c1c6557a3ce"
FORCE_RECOPY = False   # Set True to delete marker and re-copy fresh

if PROD_WORKSPACE_ID and PROD_WORKSPACE_ID != WORKSPACE_ID:
    # Discover prod bronze lakehouse
    prod_items = fabric.list_items(workspace=PROD_WORKSPACE_ID)
    prod_lh = prod_items[prod_items['Type'] == 'Lakehouse']
    prod_bronze = prod_lh[prod_lh['Display Name'].str.contains('bronze', case=False)]
    if len(prod_bronze) != 1:
        raise RuntimeError(f"Expected 1 bronze in prod, found {len(prod_bronze)}: {prod_bronze['Display Name'].tolist()}")

    prod_bronze_name = prod_bronze.iloc[0]['Display Name']
    prod_bronze_id = prod_bronze.iloc[0]['Id']

    # Get prod endpoint from lakehouse properties
    prod_props = mssparkutils.lakehouse.get(prod_bronze_name, PROD_WORKSPACE_ID)
    prod_abfs = prod_props['properties']['abfsPath']
    prod_endpoint = prod_abfs.split("@")[1].split("/")[0]

    src_root = f"abfss://{PROD_WORKSPACE_ID}@{prod_endpoint}/{prod_bronze_id}/Files"
    tgt_root = f"{bronze_root_path}/Files"

    for folder in ["SampleData", "ReferenceData"]:
        target_folder = f"{tgt_root}/"

        # --- Copy from prod ---
        print(f"\U0001f4c2 Copying {folder} from prod...")
        mssparkutils.fs.fastcp(f"{src_root}/{folder}/", target_folder)
        print(f"\u2705 {folder} copied.")

elif PROD_WORKSPACE_ID == WORKSPACE_ID:
    print("\u23ed\ufe0f PROD_WORKSPACE_ID is current workspace -- no cross-workspace copy needed.")
else:
    print("\u23ed\ufe0f PROD_WORKSPACE_ID not set -- skipping production data copy.")

### CLINICAL

In [ ]:
# ==========================================
# CONFIG
# ==========================================

base_source_path = f"{bronze_root_path}/Files/SampleData/Clinical/FHIR-NDJSON/FHIR-HDS/51KSyntheticPatients"
target_path = f"{bronze_root_path}/Files/Ingest/Clinical/FHIR-NDJSON/FHIR-HDS/51KSyntheticPatients/"

# ==========================================
# FILES TO COPY (All sample data)
# ==========================================

files_to_copy = [
    "Patient.ndjson",
    "Practitioner.ndjson",
    "PractitionerRole.ndjson",
    "Organization.ndjson",
    "Location.ndjson",
    "Encounter-1.ndjson",
    "Condition.ndjson",
    "Observation-1.ndjson",
    "Procedure-1.ndjson",
    "Claim-1.ndjson",
    "ExplanationOfBenefit-1.ndjson",
    "AllergyIntolerance.ndjson",
    "Appointment.ndjson",
    "CarePlan.ndjson",
    "CareTeam.ndjson",
    "DiagnosticReport.ndjson",
    "DocumentReference.ndjson",
    "Goal.ndjson",
    "MedicationRequest-10.ndjson",
    "RiskAssessment.ndjson",
    "Claim-2.ndjson",
    "Claim-3.ndjson",
    "Claim-4.ndjson",
    "Observation-2.ndjson",
    "Observation-3.ndjson",
    "Observation-4.ndjson",
    "Observation-5.ndjson",
    "Observation-6.ndjson",
    "Observation-7.ndjson",
    "Observation-8.ndjson",
    "Observation-9.ndjson",
    "Observation-10.ndjson",
    "Procedure-2.ndjson",
    "Procedure-3.ndjson",
    "Procedure-4.ndjson",
    "Procedure-5.ndjson",
    "Procedure-6.ndjson",
    "Procedure-7.ndjson",
    "Procedure-10.ndjson",
    "Encounter-2.ndjson",
    "Encounter-3.ndjson",
    "Encounter-4.ndjson",
    "Encounter-5.ndjson",
    "Encounter-10.ndjson",
    "ExplanationOfBenefit-2.ndjson",
    "ExplanationOfBenefit-3.ndjson",
    "ExplanationOfBenefit-4.ndjson",
    "Condition-5.ndjson",
    "Condition-10.ndjson",
    "Patient-5.ndjson",
    "Patient-10.ndjson",
    "AllergyIntolerance-5.ndjson",
    "DiagnosticReport-5.ndjson",
    "DocumentReference-10.ndjson",
    "Location.1741068704008.ndjson",
    "Organization.1741068704008.ndjson",
    "Practitioner.1741068704008.ndjson",
    "PractitionerRole.1741068704008.ndjson",
]

# ==========================================
# SENTINEL CHECK & COPY
# ==========================================

# Check if first file already exists
if file_was_copied(f"Clinical/FHIR-NDJSON/FHIR-HDS/51KSyntheticPatients/{files_to_copy[0]}"):
    print(f"\u23ed\ufe0f Clinical data already copied (found {files_to_copy[0]}) -- skipping.")
    files_to_copy = []

if files_to_copy:
    print(f"\n\U0001f4c2 Clinical FHIR -- Copying {len(files_to_copy)} files...")
    print(f"Source: {base_source_path}")
    print(f"Target: {target_path}\n")

    succeeded = []
    failed = []

    for file_name in files_to_copy:
        source_path = f"{base_source_path}/{file_name}"
        try:
            mssparkutils.fs.fastcp(source_path, target_path)
            print(f"  \u2713 {file_name}")
            succeeded.append(file_name)
        except Exception as e:
            print(f"  \u2717 {file_name}: {e}")
            failed.append(file_name)

    print(f"\n\u2713 Clinical: {len(succeeded)} copied, {len(failed)} failed.")
    if failed:
        print(f"  Failed files: {failed}")

### DICOM IMAGING

In [ ]:
# ==========================================
# CONFIG
# ==========================================

base_source_path = f"{bronze_root_path}/Files/SampleData/Imaging/DICOM/DICOM-HDS/340ImagingStudies"
target_path = f"{bronze_root_path}/Files/Ingest/Imaging/DICOM/DICOM-HDS/340ImagingStudies/"

# ==========================================
# FILES TO COPY (All imaging studies)
# ==========================================

files_to_copy = [
    "Joe-SIIM.zip",
    "Ravi-SIIM.zip",
    "Andy-SIIM.zip",
    "Sally-SIIM.zip",
    "CoherentImagingStudies.zip",
]

# ==========================================
# SENTINEL CHECK & COPY
# ==========================================

# Check if first file already exists
if file_was_copied(f"Imaging/DICOM/DICOM-HDS/340ImagingStudies/{files_to_copy[0]}"):
    print(f"\u23ed\ufe0f DICOM imaging already copied (found {files_to_copy[0]}) -- skipping.")
    files_to_copy = []

if files_to_copy:
    print(f"\n\U0001f4c2 DICOM Imaging -- Copying {len(files_to_copy)} files...")
    print(f"Source: {base_source_path}")
    print(f"Target: {target_path}\n")

    succeeded = []
    failed = []

    for file_name in files_to_copy:
        source_path = f"{base_source_path}/{file_name}"
        try:
            mssparkutils.fs.fastcp(source_path, target_path)
            print(f"  \u2713 {file_name}")
            succeeded.append(file_name)
        except Exception as e:
            print(f"  \u2717 {file_name}: {e}")
            failed.append(file_name)

    print(f"\n\u2713 DICOM: {len(succeeded)} copied, {len(failed)} failed.")
    if failed:
        print(f"  Failed files: {failed}")

### CLAIMS

In [ ]:
# Claims -- Copy all files

source_path = f"{bronze_root_path}/Files/SampleData/Claims/CCLF/CCLF-HDS/8KCCLFClaims"
target_path = f"{bronze_root_path}/Files/Ingest/Claims/CCLF/CCLF-HDS"

# Sentinel check: pick first file from source
files = mssparkutils.fs.ls(source_path)
first_file = next((f for f in files if f.isFile), None)

if first_file:
    sentinel_name = first_file.path.split("/")[-1]
    if file_was_copied(f"Claims/CCLF/CCLF-HDS/{sentinel_name}"):
        print(f"\u23ed\ufe0f Claims already copied (found {sentinel_name}) -- skipping.")
    else:
        print(f"\n\U0001f4c2 Claims -- Copying files...")
        print(f"Source: {source_path}")
        print(f"Target: {target_path}\n")

        for file_info in files:
            if file_info.isFile:
                source_file_path = file_info.path
                file_name = source_file_path.split("/")[-1]
                target_file_path = f"{target_path}/{file_name}"
                try:
                    mssparkutils.fs.cp(source_file_path, target_file_path)
                    print(f"  \u2713 {file_name}")
                except Exception as e:
                    print(f"  \u2717 Error copying {file_name}: {e}")

        print(f"\n\u2713 Claims: Done.")
else:
    print(f"\u2717 Claims: No files found at source.")

### SDOH

In [ ]:
# SDOH -- Copy all files

sdoh_csv_data_path = f"{bronze_root_path}/Files/SampleData/SDOH/CSV"
sdoh_xlsx_data_path = f"{bronze_root_path}/Files/SampleData/SDOH/XLSX"
destination_path_csv = f"{bronze_root_path}/Files/Ingest/SDOH/CSV"
destination_path_xlsx = f"{bronze_root_path}/Files/Ingest/SDOH/XLSX"

def copy_source_files_and_folders(source_path, destination_path):
    """Recursively copy files/folders, skipping existing files."""
    source_contents = mssparkutils.fs.ls(source_path)

    try:
        destination_contents = mssparkutils.fs.ls(destination_path)
        destination_files = {item.path.split("/")[-1]: item.path for item in destination_contents}
    except Exception as e:
        print(f"  Destination {destination_path} does not exist. Creating.")
        destination_files = {}
        mssparkutils.fs.mkdirs(destination_path)

    for item in source_contents:
        item_path = item.path
        item_name = item_path.split("/")[-1]
        destination_item_path = f"{destination_path}/{item_name}"

        if item.isDir:
            copy_source_files_and_folders(item_path, destination_item_path)
        else:
            if item_name in destination_files:
                print(f"  \u23ed\ufe0f Exists, skipping: {item_name}")
            else:
                print(f"  \u2713 Copying: {item_name}")
                mssparkutils.fs.cp(item_path, destination_item_path, recurse=True)

print(f"\n\U0001f4c2 SDOH -- Copying files...")
print(f"  CSV:  {sdoh_csv_data_path}")
print(f"  XLSX: {sdoh_xlsx_data_path}\n")

copy_source_files_and_folders(sdoh_csv_data_path, destination_path_csv)
copy_source_files_and_folders(sdoh_xlsx_data_path, destination_path_xlsx)

print(f"\n\u2713 SDOH: Done.")

### LOCATION

In [ ]:
# Load ZipToFipsMapping into Silver Lakehouse Tables
# Requires SDOH to be deployed (ZIPToFIPSMapping.xlsx must exist in bronze)

import pandas as pd

delta_table_path = f"{silver_root_path}/Tables/ZipToFipsMapping"

# Sentinel: check if Delta table already has data
try:
    existing = spark.read.format("delta").load(delta_table_path)
    row_count = existing.count()
    if row_count > 0:
        print(f"\u23ed\ufe0f ZipToFipsMapping already loaded ({row_count} rows) -- skipping.")
    else:
        raise Exception("empty")
except:
    excel_file_path = f"{bronze_root_path}/Files/ReferenceData/SDOH/LocationDatasets/ZIPToFIPSMapping.xlsx"

    print(f"\n\U0001f4c2 Location/Reference -- Loading ZipToFipsMapping...")
    print(f"Source: {excel_file_path}")

    try:
        df_excel = pd.read_excel(excel_file_path)
        df_excel["zip"] = df_excel["zip"].astype(str)
        df_excel["countyFips"] = df_excel["countyFips"].astype(str)

        df_spark = spark.createDataFrame(df_excel)
        df_spark.write.format("delta").mode("overwrite").save(delta_table_path)

        df_read = spark.read.format("delta").load(delta_table_path)
        print(f"  \u2713 Written {df_read.count()} rows to {delta_table_path}")
        df_read.show(5)
        print(f"\n\u2713 Location/Reference: Done.")

    except FileNotFoundError:
        print(f"  \u2717 ZIPToFIPSMapping.xlsx not found.")
        print(f"  SDOH may not be deployed in this workspace.")
        print(f"  Run this cell again after deploying SDOH.")